# Shapley Methods — Real Data Summary (Sachs Cell Signalling)

Analysis of Shapley attribution methods on the **Sachs et al. (2005)** protein signalling dataset.  
Reuses helper functions from `analysis_utils.py` for metric consistency with synthetic data analysis.

**Dataset:** `sachs` — 7,466 observations of 11 proteins, target = `akt` (Akt kinase)

| Property | Value |
|---------|-------|
| X features | 10 proteins: raf, mek, plc, pip2, pip3, erk, pka, pkc, p38, jnk |
| Y target | akt |
| True Y-parents | 3: pip3, pka, erk (from Sachs 2005 reference DAG) |
| Sample size | 5,972 train / 1,494 test |
| Model | LightGBM (R² = 0.68 on test) |

**Metrics computed:**
- **Spearman ρ** vs Traditional — feature ranking correlation
- **Top-K Jaccard** vs Traditional — feature set overlap at K = 5, 10, 20
- **True-parent Precision@K** — fraction of top-K that are true Y-parents
- **GSS** — Graph Sensitivity Score (PC vs LiNGAM magnitude shift)
- **SSS** — Sign Stability Score (PC vs LiNGAM sign agreement)
- **Sign Alignment** vs True graph — fraction of instances with matching sign
- **Magnitude TGA** vs True graph — relative magnitude deviation

In [1]:
import sys
import importlib
from pathlib import Path as _Path
sys.path.insert(0, str(_Path("..").resolve()))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import json
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

from scipy.stats import spearmanr

# Force reload so any changes to analysis_utils.py are picked up
if "analysis_utils" in sys.modules:
    importlib.reload(sys.modules["analysis_utils"])

from analysis_utils import (
    mean_abs_shap, top_k_jaccard,
    compute_sign_alignment, compute_tga, compute_sss,
    compute_gss, top_k_features, rank_shift_top_k,
    flow_graph_stats,
    build_dag_graph, make_dag_pos, draw_dag_highlight,
)

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_DIR           = Path("..").resolve()
EXPLAINABILITY_DIR = BASE_DIR / "data" / "explainability"
CAUSAL_DIR         = BASE_DIR / "data" / "causal"
SYNTHETIC_DIR      = BASE_DIR / "data" / "synthetic"

# ── Dataset configuration ─────────────────────────────────────────────────────
DATASET      = "sachs"
MODEL        = "lgbm"
K_VALUES     = [3, 5, 10]  # Adjusted for smaller feature set (10 features)
BASE_METHODS = ["Asymmetric", "Causal", "Flow"]
DISC_GRAPHS  = ["PC", "LiNGAM"]

METHOD_COLORS = {
    "Scratch":    "#636363",
    "Asymmetric": "#2166ac",
    "Causal":     "#4dac26",
    "Flow":       "#d01c8b",
}

print(f"Dataset: {DATASET}")
print(f"Model: {MODEL}")
print(f"K values: {K_VALUES}")

Dataset: sachs
Model: lgbm
K values: [3, 5, 10]


## 1. Load Data

Load Shapley values, causal graphs (PC, LiNGAM, True), metadata, and true Y-parent set.

In [2]:
def load_real_dataset(dataset: str, model: str = "lgbm") -> dict:
    """
    Load all Shapley values, causal graphs, metadata, and true-graph arrays
    for the real dataset. Returns a dict with all data needed for metric computation.
    """
    exp_base = EXPLAINABILITY_DIR / dataset / model
    causal   = CAUSAL_DIR

    # ── Main SHAP data (from pipeline output) ─────────────────────────────
    keys = {
        "Scratch":             exp_base / "scratch"  / "shapley_values.npy",
        "Asymmetric (PC)":     exp_base / "pc"       / "asymmetric" / "shapley_values.npy",
        "Causal (PC)":         exp_base / "pc"       / "causal"     / "shapley_values.npy",
        "Flow (PC)":           exp_base / "pc"       / "flow"       / "shapley_values.npy",
        "Asymmetric (LiNGAM)": exp_base / "lingam"   / "asymmetric" / "shapley_values.npy",
        "Causal (LiNGAM)":     exp_base / "lingam"   / "causal"     / "shapley_values.npy",
        "Flow (LiNGAM)":       exp_base / "lingam"   / "flow"       / "shapley_values.npy",
    }
    subset_keys = {
        "Asymmetric (PC)":     exp_base / "pc"       / "asymmetric" / "shapley_values.npy",
        "Causal (PC)":         exp_base / "pc"       / "causal"     / "shapley_values.npy",
        "Flow (PC)":           exp_base / "pc"       / "flow"       / "shapley_values.npy",
        "Asymmetric (LiNGAM)": exp_base / "lingam"   / "asymmetric" / "shapley_values.npy",
        "Causal (LiNGAM)":     exp_base / "lingam"   / "causal"     / "shapley_values.npy",
        "Flow (LiNGAM)":       exp_base / "lingam"   / "flow"       / "shapley_values.npy",
        "Asymmetric (True)":   exp_base / "true"     / "asymmetric" / "shapley_values.npy",
        "Causal (True)":       exp_base / "true"     / "causal"     / "shapley_values.npy",
        "Flow (True)":         exp_base / "true"     / "flow"       / "shapley_values.npy",
    }

    shap_data        = {k: np.load(v) for k, v in keys.items()        if v.exists()}
    subset_shap_data = {k: np.load(v) for k, v in subset_keys.items() if v.exists()}

    # ── Feature names ──────────────────────────────────────────────────────
    with open(causal / f"{dataset}_pc_results.json") as f:
        pc_results = json.load(f)
    with open(causal / f"{dataset}_lingam_results.json") as f:
        lingam_results = json.load(f)
    feature_names = [n for n in pc_results["feature_names"] if n != "Y"]

    # ── True Y-parents ─────────────────────────────────────────────────────
    with open(SYNTHETIC_DIR / f"{dataset}_metadata.json") as f:
        meta = json.load(f)
    y_parent_set = set(meta["y_parent_indices"])

    # ── Graph stats ────────────────────────────────────────────────────────
    pc_edges,     pc_sources     = flow_graph_stats(pc_results["adjacency_matrix"],     len(feature_names))
    lingam_edges, lingam_sources = flow_graph_stats(lingam_results["adjacency_matrix"], len(feature_names))

    return dict(
        shap_data        = shap_data,
        subset_shap_data = subset_shap_data,
        feature_names    = feature_names,
        y_parent_set     = y_parent_set,
        pc_edges         = pc_edges,
        lingam_edges     = lingam_edges,
    )


# Load dataset
data = load_real_dataset(DATASET)
n_shap = data["shap_data"]["Scratch"].shape[0]
n_feat = len(data["feature_names"])
n_par  = len(data["y_parent_set"])
loaded = len(data["shap_data"])
sub_loaded = len(data["subset_shap_data"])

print(f"\n{'=' * 72}")
print(f"  Loaded: {loaded} main SHAP arrays + {sub_loaded} subset arrays")
print(f"  Instances: {n_shap}")
print(f"  Features: {n_feat}")
print(f"  True Y-parents: {n_par} — {sorted([data['feature_names'][i] for i in data['y_parent_set']])}")
print(f"  PC edges: {data['pc_edges']}")
print(f"  LiNGAM edges: {data['lingam_edges']}")
print(f"{'=' * 72}")


  Loaded: 7 main SHAP arrays + 9 subset arrays
  Instances: 100
  Features: 10
  True Y-parents: 10 — ['erk', 'jnk', 'mek', 'p38', 'pip2', 'pip3', 'pka', 'pkc', 'plc', 'raf']
  PC edges: 29
  LiNGAM edges: 36


## 2. Compute All Metrics

For each method × graph, compute Spearman ρ, Top-K Jaccard, Precision@K, GSS, SSS, Sign Alignment, and TGA.

In [3]:
all_metrics = []   # list of dicts, one per (method, graph)
gss_metrics = []   # one per method
sss_metrics = []   # one per method
tga_metrics = []   # one per (method, graph)
sa_metrics  = []   # sign alignment, one per (method, graph)
sa_scratch_metrics  = []   # sign alignment vs Scratch
tga_scratch_metrics = []   # TGA vs Scratch

shap_data     = data["shap_data"]
subset_shap   = data["subset_shap_data"]
feature_names = data["feature_names"]
y_parent_set  = data["y_parent_set"]
n_feat        = len(feature_names)
random_base   = len(y_parent_set) / n_feat

scratch_ma = mean_abs_shap(shap_data, "Scratch")

# ── Spearman ρ, Jaccard, Precision@K ──────────────────────────────────────
for meth in BASE_METHODS:
    for g in DISC_GRAPHS:
        key = f"{meth} ({g})"
        if key not in shap_data:
            continue
        ma = mean_abs_shap(shap_data, key)
        rho, _ = spearmanr(ma, scratch_ma)
        for k in K_VALUES:
            jacc = top_k_jaccard(ma, scratch_ma, k)
            top_k_idx = set(np.argsort(ma)[-k:])
            prec = len(top_k_idx & y_parent_set) / k
            all_metrics.append(dict(
                Method      = meth,
                Graph       = g,
                K           = k,
                Spearman    = float(rho),
                Jaccard     = float(jacc),
                Precision   = float(prec),
                RandomBase  = float(random_base),
            ))

# ── Scratch Precision@K (reference) ───────────────────────────────────────
scratch_ma_arr = mean_abs_shap(shap_data, "Scratch")
for k in K_VALUES:
    top_k_idx = set(np.argsort(scratch_ma_arr)[-k:])
    prec = len(top_k_idx & y_parent_set) / k
    all_metrics.append(dict(
        Method="Scratch", Graph="—",
        K=k, Spearman=np.nan, Jaccard=np.nan, Precision=float(prec),
        RandomBase=float(random_base),
    ))

# ── GSS — using standard compute_gss function ─────────────────────────────
# Returns per-feature arrays: dict[method] -> ndarray(n_features,)
gss_feat = compute_gss(shap_data, BASE_METHODS)
for meth, feat_arr in gss_feat.items():
    gss_metrics.append(dict(
        Method    = meth,
        MeanGSS   = float(np.mean(feat_arr)),
        MedianGSS = float(np.median(feat_arr)),
    ))

# ── SSS — using standard compute_sss function ─────────────────────────────
# Returns per-feature arrays: dict[method] -> ndarray(n_features,)
sss_feat = compute_sss(shap_data, BASE_METHODS)
for meth, feat_arr in sss_feat.items():
    sss_metrics.append(dict(
        Method  = meth,
        MeanSSS = float(np.nanmean(feat_arr)),
    ))

# ── Sign Alignment vs True ─────────────────────────────────────────────────
# Returns per-feature arrays: dict[(method, graph)] -> ndarray(n_features,)
sign_align = compute_sign_alignment(subset_shap, BASE_METHODS, DISC_GRAPHS, reference="True")
for (meth, g), arr in sign_align.items():
    sa_metrics.append(dict(
        Method        = meth,
        Graph         = g,
        MeanSignAlign = float(np.nanmean(arr)),
    ))

# ── Magnitude TGA vs True ──────────────────────────────────────────────────
# Returns per-feature arrays: dict[(method, graph)] -> ndarray(n_features,)
tga_data, _ = compute_tga(subset_shap, feature_names, BASE_METHODS, DISC_GRAPHS, reference="True")
for (meth, g), arr in tga_data.items():
    tga_metrics.append(dict(
        Method  = meth,
        Graph   = g,
        MeanTGA = float(np.mean(arr)),
    ))

# ── Sign Alignment vs Scratch (Traditional baseline) ──────────────────────
# Merge subset SHAP + Scratch as reference
combined = dict(subset_shap)
if "Scratch" in shap_data:
    combined["Scratch"] = shap_data["Scratch"]

sign_align_scratch = compute_sign_alignment(
    combined, BASE_METHODS, DISC_GRAPHS, reference="Scratch"
)
for (meth, g), arr in sign_align_scratch.items():
    sa_scratch_metrics.append(dict(
        Method        = meth,
        Graph         = g,
        MeanSignAlign = float(np.nanmean(arr)),
    ))

# ── Magnitude TGA vs Scratch (Traditional baseline) ───────────────────────
tga_scratch, _ = compute_tga(
    combined, feature_names, BASE_METHODS, DISC_GRAPHS, reference="Scratch"
)
for (meth, g), arr in tga_scratch.items():
    tga_scratch_metrics.append(dict(
        Method  = meth,
        Graph   = g,
        MeanTGA = float(np.nanmean(arr)),
    ))

# ── Build summary DataFrames ───────────────────────────────────────────────
df_metrics     = pd.DataFrame(all_metrics)
df_gss         = pd.DataFrame(gss_metrics)
df_sss         = pd.DataFrame(sss_metrics)
df_sa          = pd.DataFrame(sa_metrics)
df_tga         = pd.DataFrame(tga_metrics)
df_sa_scratch  = pd.DataFrame(sa_scratch_metrics)
df_tga_scratch = pd.DataFrame(tga_scratch_metrics)

print(f"Metrics computed:")
print(f"  All metrics: {len(df_metrics)} rows")
print(f"  GSS: {len(df_gss)} rows (per-feature arrays stored in gss_feat)")
print(f"  SSS: {len(df_sss)} rows (per-feature arrays stored in sss_feat)")
print(f"  Sign Alignment vs True: {len(df_sa)} rows")
print(f"  TGA vs True: {len(df_tga)} rows")
print(f"  Sign Alignment vs Scratch: {len(df_sa_scratch)} rows")
print(f"  TGA vs Scratch: {len(df_tga_scratch)} rows")

Metrics computed:
  All metrics: 21 rows
  GSS: 3 rows (per-feature arrays stored in gss_feat)
  SSS: 3 rows (per-feature arrays stored in sss_feat)
  Sign Alignment vs True: 6 rows
  TGA vs True: 6 rows
  Sign Alignment vs Scratch: 6 rows
  TGA vs Scratch: 6 rows


## 3. Feature Profile Function

Detailed diagnostic for any feature under any method/graph combination.

In [4]:
def feature_profile(feature_name: str, method_key: str):
    """
    Print a full metric profile for one feature under one method/graph combination
    and return a rank-comparison DataFrame across all loaded methods.

    Parameters
    ----------
    feature_name : str   Feature name, e.g. "raf", "pip3"
    method_key   : str   Method key, e.g. "Asymmetric (PC)", "Causal (LiNGAM)", "Scratch"

    Returns
    -------
    rank_df : pd.DataFrame
        Rank of this feature across all loaded method keys.
    """
    # ── Parse base method and graph from key ─────────────────────────────────
    if " (" in method_key and method_key.endswith(")"):
        base_meth = method_key[: method_key.rfind(" (")]
        graph     = method_key[method_key.rfind("(") + 1 : -1]
    else:
        base_meth = method_key
        graph     = None

    W = 72

    if feature_name not in feature_names:
        print(f"Feature '{feature_name}' not found. Available: {feature_names}")
        return pd.DataFrame()

    feat_idx = feature_names.index(feature_name)
    arr_full = shap_data.get(method_key)
    if arr_full is None:
        arr_full = subset_shap.get(method_key)

    if arr_full is None:
        all_keys = list(shap_data) + [k for k in subset_shap if k not in shap_data]
        print(f"Method key '{method_key}' not found. Available: {sorted(all_keys)}")
        return pd.DataFrame()

    # ── Core attribution ──────────────────────────────────────────────────
    feat_shap   = arr_full[:, feat_idx]
    mean_signed = float(feat_shap.mean())
    mean_abs    = float(np.abs(feat_shap).mean())
    is_y_parent = feat_idx in y_parent_set

    ma_all    = np.abs(arr_full).mean(axis=0)
    rank_this = int(np.argsort(np.argsort(-ma_all))[feat_idx]) + 1

    # Rank in Scratch
    rank_scratch = None
    if "Scratch" in shap_data:
        scratch_ma   = np.abs(shap_data["Scratch"]).mean(axis=0)
        rank_scratch = int(np.argsort(np.argsort(-scratch_ma))[feat_idx]) + 1

    delta_rank = (rank_this - rank_scratch) if rank_scratch is not None else None

    # ── Graph Sensitivity (GSS + SSS) ─────────────────────────────────────
    gss_val = sss_val = None
    if base_meth in BASE_METHODS:
        gss_map = compute_gss(shap_data, [base_meth])
        if base_meth in gss_map:
            gss_val = float(gss_map[base_meth][feat_idx])
        sss_map = compute_sss(shap_data, [base_meth])
        if base_meth in sss_map:
            sss_val = float(sss_map[base_meth][feat_idx])

    # ── Alignment vs True DAG ─────────────────────────────────────────────
    sa_true_val = tga_true_val = None
    if graph and graph not in ("Scratch",) and base_meth in BASE_METHODS:
        sa_true_map = compute_sign_alignment(
            subset_shap, [base_meth], [graph], reference="True")
        tga_true_map, _ = compute_tga(
            subset_shap, feature_names, [base_meth], [graph], reference="True")
        if (base_meth, graph) in sa_true_map:
            sa_true_val  = float(sa_true_map[(base_meth, graph)][feat_idx])
        if (base_meth, graph) in tga_true_map:
            tga_true_val = float(tga_true_map[(base_meth, graph)][feat_idx])

    # ── Alignment vs Scratch ───────────────────────────────────────────────
    sa_scratch_val = tga_scratch_val = None
    if graph and base_meth in BASE_METHODS:
        combined_d = {**subset_shap}
        if "Scratch" in shap_data:
            combined_d["Scratch"] = shap_data["Scratch"]
        sa_scr_map = compute_sign_alignment(
            combined_d, [base_meth], [graph], reference="Scratch")
        tga_scr_map, _ = compute_tga(
            combined_d, feature_names, [base_meth], [graph], reference="Scratch")
        if (base_meth, graph) in sa_scr_map:
            sa_scratch_val  = float(sa_scr_map[(base_meth, graph)][feat_idx])
        if (base_meth, graph) in tga_scr_map:
            tga_scratch_val = float(tga_scr_map[(base_meth, graph)][feat_idx])

    # ── Rank comparison across all loaded method keys ─────────────────────
    all_keys = list(shap_data.keys()) + [k for k in subset_shap if k not in shap_data]
    rank_rows = []
    for k in sorted(all_keys):
        src    = shap_data if k in shap_data else subset_shap
        arr_k  = src[k]
        ma_k   = np.abs(arr_k).mean(axis=0)
        ms_k   = arr_k[:, feat_idx].mean()
        rk     = int(np.argsort(np.argsort(-ma_k))[feat_idx]) + 1
        rank_rows.append({
            "Method":       k,
            "Rank":         rk,
            "Mean SHAP":    round(float(ms_k), 6),
            "Mean |SHAP|":  round(float(ma_k[feat_idx]), 6),
            "Current →":    "←" if k == method_key else "",
        })
    rank_df = (pd.DataFrame(rank_rows)
                 .sort_values("Mean |SHAP|", ascending=False)
                 .reset_index(drop=True))
    rank_df.index += 1

    # ── Print ─────────────────────────────────────────────────────────────
    print(f"\n{'═' * W}")
    print(f"  FEATURE PROFILE: {feature_name}  |  {method_key}")
    print(f"{'═' * W}")

    print(f"\n  CORE ATTRIBUTION")
    print(f"    Mean SHAP (signed)  :  {mean_signed:+.6f}")
    print(f"    Mean |SHAP|         :   {mean_abs:.6f}  →  Rank {rank_this}/{n_feat}")
    print(f"    Is Y-parent (true)  :   {'Yes ✓' if is_y_parent else 'No'}")
    if rank_scratch is not None:
        sign = "+" if delta_rank > 0 else ""
        print(f"    Rank vs Scratch     :   {rank_scratch}  →  {rank_this}"
              f"  ({sign}{delta_rank}  {'↓ less important' if delta_rank > 0 else '↑ more important' if delta_rank < 0 else '= no change'})")

    if gss_val is not None or sss_val is not None:
        print(f"\n  GRAPH SENSITIVITY  ({base_meth}: PC vs LiNGAM)")
        if gss_val is not None:
            dir_str = "PC assigns more weight" if gss_val > 0 else "LiNGAM assigns more weight"
            print(f"    GSS (magnitude)    :  {gss_val:+.6f}  ({dir_str})")
        if sss_val is not None:
            stab = "stable" if sss_val >= 0.8 else ("unstable" if sss_val < 0.5 else "moderate")
            print(f"    SSS (sign)         :   {sss_val:.4f}  ({stab})")

    if sa_true_val is not None or tga_true_val is not None:
        print(f"\n  ALIGNMENT VS TRUE DAG  ({base_meth} — {graph})")
        if sa_true_val is not None:
            agree = "agrees" if sa_true_val >= 0.8 else ("disagrees" if sa_true_val < 0.5 else "partially agrees")
            print(f"    Sign Alignment     :   {sa_true_val:.4f}  ({agree} with true oracle)")
        if tga_true_val is not None:
            oe = "over-estimates" if tga_true_val > 0 else "under-estimates"
            print(f"    TGA                :  {tga_true_val:+.6f}  ({oe} true-graph magnitude)")

    if sa_scratch_val is not None or tga_scratch_val is not None:
        print(f"\n  ALIGNMENT VS SCRATCH  ({base_meth} — {graph})")
        if sa_scratch_val is not None:
            agree = "agrees" if sa_scratch_val >= 0.8 else ("disagrees" if sa_scratch_val < 0.5 else "partially agrees")
            print(f"    Sign Alignment     :   {sa_scratch_val:.4f}  ({agree} with scratch baseline)")
        if tga_scratch_val is not None:
            oe = "over-estimates" if tga_scratch_val > 0 else "under-estimates"
            print(f"    TGA                :  {tga_scratch_val:+.6f}  ({oe} scratch magnitude)")

    print(f"\n  RANK COMPARISON  (all methods, feature: {feature_name})")
    display(rank_df)

    return rank_df


# ── Example usage ─────────────────────────────────────────────────────────────
# feature_profile("pip3", "Asymmetric (PC)")
# feature_profile("erk", "Causal (LiNGAM)")

## 4. Ranking Correlation & Feature Set Overlap

### 4a — Spearman ρ vs Traditional  
Feature ranking correlation between each causal method and the graph-free Traditional baseline.

### 4b — Top-K Jaccard vs Traditional  
Fraction of the top-K features shared with Traditional's top-K set.

### 4c — True-Parent Precision@K  
Fraction of the top-K features that are true causal parents of Y (pip3, pka, erk).

In [5]:
# ── 4a: Spearman ρ bar chart — method × graph ────────────────────────────────
df_rho = df_metrics[df_metrics["Method"].isin(BASE_METHODS)].drop_duplicates(
    subset=["Method", "Graph", "Spearman"]
)[["Method", "Graph", "Spearman"]].dropna()
df_rho = df_rho.groupby(["Method", "Graph"], as_index=False)["Spearman"].first()

fig_rho = go.Figure()
for meth in BASE_METHODS:
    sub = df_rho[df_rho["Method"] == meth]
    fig_rho.add_trace(go.Bar(
        name         = meth,
        x            = sub["Graph"].tolist(),
        y            = sub["Spearman"].tolist(),
        marker_color = METHOD_COLORS[meth],
        hovertemplate=f"<b>{meth}</b><br>Graph=%{{x}}<br>ρ=%{{y:.3f}}<extra></extra>",
    ))

fig_rho.update_layout(
    title=dict(text="Spearman ρ vs Traditional — Sachs Real Data<br>"
               "<sup>1.0 = same feature ranking as graph-free baseline</sup>",
               font=dict(size=13)),
    barmode="group",
    yaxis=dict(title="Spearman ρ", range=[0, 1.05]),
    height=360, width=600,
    legend=dict(x=0.01, y=0.99, xanchor="left", font=dict(size=11)),
    margin=dict(l=60, r=30, t=80, b=60),
)
fig_rho.show()

In [6]:
# ── 4b: Top-K Jaccard heatmap — method × graph vs K ──────────────────────────
df_jacc = df_metrics[df_metrics["Method"].isin(BASE_METHODS)][["Method", "Graph", "K", "Jaccard"]]

pivot_jacc = df_jacc.pivot_table(
    index=["Method", "Graph"], columns="K", values="Jaccard"
)
row_labels = [f"{m} ({g})" for m, g in pivot_jacc.index]
col_labels = [f"K={k}" for k in pivot_jacc.columns]

fig_jacc = go.Figure(go.Heatmap(
    z            = pivot_jacc.values,
    x            = col_labels,
    y            = row_labels,
    colorscale   = "YlGnBu",
    zmin=0, zmax=1,
    text         = [[f"{v:.3f}" for v in row] for row in pivot_jacc.values],
    texttemplate = "%{text}",
    textfont     = dict(size=12),
    colorbar     = dict(title=dict(text="Jaccard", font=dict(size=11))),
    hovertemplate="<b>%{y}</b> — %{x}<br>Jaccard=%{z:.3f}<extra></extra>",
))
fig_jacc.update_layout(
    title=dict(text="Top-K Jaccard vs Traditional — Sachs Real Data<br>"
               "<sup>Fraction of top-K features shared with Traditional</sup>",
               font=dict(size=13)),
    height=320, width=600,
    margin=dict(l=160, r=80, t=80, b=60),
    xaxis=dict(tickfont=dict(size=11)),
    yaxis=dict(tickfont=dict(size=10)),
)
fig_jacc.show()

In [7]:
# ── 4c: True-parent Precision@K — one subplot per K-value ───────────────────
fig_prec = make_subplots(
    rows=1, cols=len(K_VALUES),
    subplot_titles=[f"Precision@{k}" for k in K_VALUES],
    shared_yaxes=True,
)
LEGEND_SEEN = set()
combos = [(m, g) for m in BASE_METHODS for g in DISC_GRAPHS]

for ci, k in enumerate(K_VALUES, start=1):
    sub = df_metrics[(df_metrics["K"] == k) & (df_metrics["Method"].isin(BASE_METHODS))]
    for meth, g in combos:
        rows = sub[(sub["Method"] == meth) & (sub["Graph"] == g)]
        if rows.empty:
            continue
        name = f"{meth} ({g})"
        show = name not in LEGEND_SEEN
        LEGEND_SEEN.add(name)
        fig_prec.add_trace(
            go.Bar(
                x           = [g],
                y           = [rows.iloc[0]["Precision"]],
                name        = name,
                legendgroup = name,
                showlegend  = show,
                marker_color= METHOD_COLORS[meth],
                opacity     = 0.7 if g == "LiNGAM" else 1.0,
                hovertemplate=f"<b>{name}</b><br>Prec=%{{y:.3f}}<extra></extra>",
            ),
            row=1, col=ci,
        )
    # Scratch precision
    scratch_rows = df_metrics[(df_metrics["K"] == k) & (df_metrics["Method"] == "Scratch")]
    if not scratch_rows.empty:
        fig_prec.add_trace(
            go.Bar(
                x=["—"], y=[scratch_rows.iloc[0]["Precision"]],
                name="Scratch", legendgroup="Scratch",
                showlegend=(ci == 1),
                marker_color=METHOD_COLORS["Scratch"],
                hovertemplate="<b>Scratch</b><br>Prec=%{y:.3f}<extra></extra>",
            ),
            row=1, col=ci,
        )
    # Random baseline line
    random_val = random_base
    fig_prec.add_trace(
        go.Scatter(
            x=["PC", "LiNGAM", "—"], y=[random_val]*3,
            mode="lines", line=dict(dash="dash", color="black", width=1.2),
            name="Random", legendgroup="Random", showlegend=(ci == 1),
            hovertemplate="Random baseline<br>Prec=%{y:.3f}<extra></extra>",
        ),
        row=1, col=ci,
    )

fig_prec.update_layout(
    title=dict(text="True-Parent Precision@K — Sachs Real Data<br>"
               "<sup>Dashed line = random baseline (3 true parents / 10 features = 0.30)</sup>",
               font=dict(size=13)),
    height=360, width=800,
    barmode="group",
    legend=dict(x=1.01, y=1, xanchor="left", font=dict(size=10)),
    margin=dict(l=60, r=200, t=80, b=80),
)
fig_prec.update_yaxes(range=[0, 1.05], title_text="Precision", col=1)
fig_prec.show()

## 5. Graph Sensitivity Score (GSS)

How much does each method's output shift between the PC and LiNGAM graphs?  
Signed: positive = PC assigns higher importance, negative = LiNGAM higher.

In [8]:
# GSS bar chart — method comparison
fig_gss = go.Figure()
for meth in BASE_METHODS:
    sub = df_gss[df_gss["Method"] == meth]
    if sub.empty:
        continue
    fig_gss.add_trace(go.Bar(
        name         = meth,
        x            = [meth],
        y            = [sub.iloc[0]["MeanGSS"]],
        marker_color = METHOD_COLORS[meth],
        hovertemplate=f"<b>{meth}</b><br>MeanGSS=%{{y:.4f}}<extra></extra>",
    ))

fig_gss.update_layout(
    title=dict(text="Graph Sensitivity Score (Mean GSS) — Sachs Real Data<br>"
               "<sup>Signed: positive = PC assigns higher importance, negative = LiNGAM higher</sup>",
               font=dict(size=13)),
    yaxis=dict(title="Mean GSS", zeroline=True, zerolinewidth=1.5, zerolinecolor="gray"),
    height=380, width=600,
    showlegend=False,
    margin=dict(l=70, r=30, t=90, b=80),
)
fig_gss.show()

## 6. Sign Stability Score (SSS) & Cross-Evaluation

**SSS** measures sign agreement between PC and LiNGAM for each feature.  
Higher SSS = signs are more consistent between the two discovered graphs (more stable).

The **cross-evaluation scatter** places every method in a common stability space:
- **x-axis — GSS**: signed magnitude bias between PC and LiNGAM
- **y-axis — (1 − SSS)**: sign flip rate between PC and LiNGAM

In [9]:
# ── SSS bar chart ─────────────────────────────────────────────────────────────
fig_sss = go.Figure()
for meth in BASE_METHODS:
    sub = df_sss[df_sss["Method"] == meth]
    if sub.empty:
        continue
    fig_sss.add_trace(go.Bar(
        name         = meth,
        x            = [meth],
        y            = [sub.iloc[0]["MeanSSS"]],
        marker_color = METHOD_COLORS[meth],
        hovertemplate=f"<b>{meth}</b><br>MeanSSS=%{{y:.3f}}<extra></extra>",
    ))

fig_sss.update_layout(
    title=dict(text="Sign Stability Score (Mean SSS) — Sachs Real Data<br>"
               "<sup>Fraction of (instance, feature) pairs where SHAP sign agrees between PC and LiNGAM</sup>",
               font=dict(size=13)),
    yaxis=dict(title="Mean SSS (sign agreement)", range=[0, 1.05]),
    height=380, width=600,
    showlegend=False,
    margin=dict(l=70, r=30, t=90, b=80),
)
fig_sss.show()

In [10]:
# ── Cross-evaluation: GSS vs (1 − SSS) scatter ────────────────────────────────
df_cross = df_gss[["Method", "MeanGSS"]].merge(
    df_sss[["Method", "MeanSSS"]],
    on="Method",
)
df_cross["SignFlipRate"] = 1 - df_cross["MeanSSS"]

fig_cross = go.Figure()
for _, row in df_cross.iterrows():
    meth = row["Method"]
    fig_cross.add_trace(go.Scatter(
        x    = [row["MeanGSS"]],
        y    = [row["SignFlipRate"]],
        mode = "markers+text",
        name = meth,
        text = [meth],
        textposition = "top center",
        marker = dict(
            color  = METHOD_COLORS[meth],
            symbol = "circle",
            size   = 13,
            line   = dict(width=1.2, color="white"),
        ),
        hovertemplate=(
            f"<b>{meth}</b><br>"
            "GSS (signed magnitude bias): %{x:.4f}<br>"
            "Sign flip rate (1−SSS): %{y:.3f}<extra></extra>"
        ),
    ))

# Symmetric x-axis range around 0
gss_abs_max = df_cross["MeanGSS"].abs().max() * 1.3
y_max       = df_cross["SignFlipRate"].max() * 1.2

# Vertical reference line at x=0
fig_cross.add_vline(
    x=0, line=dict(dash="dot", color="lightgray", width=1.5)
)
fig_cross.add_annotation(
    x=0, y=y_max * 0.97, text="PC = LiNGAM", showarrow=False,
    font=dict(size=9, color="#aaaaaa"), xanchor="center",
)

# Quadrant labels
for qx, qy, label in [
    (-gss_abs_max * 0.6, y_max * 0.75, "LiNGAM > PC<br>Sign-unstable"),
    ( gss_abs_max * 0.6, y_max * 0.75, "PC > LiNGAM<br>Sign-unstable"),
    (-gss_abs_max * 0.6, y_max * 0.15, "LiNGAM > PC<br>Sign-stable"),
    ( gss_abs_max * 0.6, y_max * 0.15, "PC > LiNGAM<br>Sign-stable"),
]:
    fig_cross.add_annotation(
        x=qx, y=qy, text=label, showarrow=False,
        font=dict(size=9, color="#bbbbbb"), align="center",
    )

fig_cross.update_layout(
    title=dict(
        text="GSS vs Sign Flip Rate (1 − SSS) — Sachs Real Data<br>"
             "<sup>x>0: PC assigns higher importance; x<0: LiNGAM higher</sup>",
        font=dict(size=12),
    ),
    xaxis=dict(
        title="GSS — Signed Magnitude Bias",
        range=[-gss_abs_max, gss_abs_max],
        zeroline=True, zerolinewidth=1.5, zerolinecolor="gray",
    ),
    yaxis=dict(
        title="1 − SSS — Sign Flip Rate",
        range=[0, y_max],
        zeroline=False,
    ),
    height=480, width=660,
    showlegend=False,
    margin=dict(l=70, r=50, t=90, b=70),
)
fig_cross.show()

In [11]:
# ── Summary Table: Mean Metrics by Method (GSS vs 1-SSS) ────────────────────
df_cross_summary = df_cross[["Method", "MeanGSS", "MeanSSS", "SignFlipRate"]].copy()
df_cross_summary = df_cross_summary.rename(columns={
    "MeanGSS": "GSS (Signed Magnitude Bias)",
    "MeanSSS": "SSS (Sign Stability)",
    "SignFlipRate": "Sign Flip Rate (1−SSS)"
})

print("\nSummary: PC vs LiNGAM Stability Metrics — Sachs Real Data")
print("=" * 80)
display(df_cross_summary.round(4))


Summary: PC vs LiNGAM Stability Metrics — Sachs Real Data


,Method,GSS (Signed Magnitude Bias),SSS (Sign Stability),Sign Flip Rate (1−SSS)
0,Asymmetric,0.1775,0.8140,0.1860
1,Causal,-0.7182,0.6820,0.3180
2,Flow,0.4141,0.7812,0.2188


In [12]:
# ── Feature-level: GSS vs (1 − SSS) per feature ───────────────────────────────
# Instead of one point per method (mean), show one point per feature per method.
# This reveals the distribution and any clustering/separation patterns.

fig_cross_feat = go.Figure()

for meth in BASE_METHODS:
    if meth not in gss_feat or meth not in sss_feat:
        continue
    
    # Get per-feature arrays
    gss_arr = gss_feat[meth]  # shape: (n_features,)
    sss_arr = sss_feat[meth]  # shape: (n_features,)
    sign_flip_arr = 1 - sss_arr
    
    # Create scatter trace with all features
    fig_cross_feat.add_trace(go.Scatter(
        x=gss_arr,
        y=sign_flip_arr,
        mode="markers",
        name=meth,
        marker=dict(
            color=METHOD_COLORS[meth],
            size=8,
            opacity=0.6,
            line=dict(width=0.5, color="white"),
        ),
        hovertemplate=(
            f"<b>{meth}</b><br>"
            "Feature: %{text}<br>"
            "GSS: %{x:.4f}<br>"
            "Sign flip rate: %{y:.3f}<extra></extra>"
        ),
        text=feature_names,  # Show feature names on hover
    ))

# Compute axis ranges based on all feature data
all_gss_values = np.concatenate([gss_feat[m] for m in BASE_METHODS if m in gss_feat])
all_sign_flip = np.concatenate([1 - sss_feat[m] for m in BASE_METHODS if m in sss_feat])
gss_abs_max_feat = np.abs(all_gss_values).max() * 1.3
y_max_feat = all_sign_flip.max() * 1.2

# Vertical reference line at x=0
fig_cross_feat.add_vline(
    x=0, line=dict(dash="dot", color="lightgray", width=1.5)
)
fig_cross_feat.add_annotation(
    x=0, y=y_max_feat * 0.97, text="PC = LiNGAM", showarrow=False,
    font=dict(size=9, color="#aaaaaa"), xanchor="center",
)

# Quadrant labels
for qx, qy, label in [
    (-gss_abs_max_feat * 0.6, y_max_feat * 0.75, "LiNGAM > PC<br>Sign-unstable"),
    ( gss_abs_max_feat * 0.6, y_max_feat * 0.75, "PC > LiNGAM<br>Sign-unstable"),
    (-gss_abs_max_feat * 0.6, y_max_feat * 0.15, "LiNGAM > PC<br>Sign-stable"),
    ( gss_abs_max_feat * 0.6, y_max_feat * 0.15, "PC > LiNGAM<br>Sign-stable"),
]:
    fig_cross_feat.add_annotation(
        x=qx, y=qy, text=label, showarrow=False,
        font=dict(size=9, color="#bbbbbb"), align="center",
    )

fig_cross_feat.update_layout(
    title=dict(
        text="GSS vs Sign Flip Rate (1 − SSS) — Per-Feature Distribution — Sachs Real Data<br>"
             "<sup>Each dot = one feature. Shows within-method variability. "
             "x>0: PC assigns higher importance; x<0: LiNGAM higher</sup>",
        font=dict(size=12),
    ),
    xaxis=dict(
        title="GSS — Signed Magnitude Bias (per feature)",
        range=[-gss_abs_max_feat, gss_abs_max_feat],
        zeroline=True, zerolinewidth=1.5, zerolinecolor="gray",
    ),
    yaxis=dict(
        title="1 − SSS — Sign Flip Rate (per feature)",
        range=[0, y_max_feat],
        zeroline=False,
    ),
    height=480, width=700,
    legend=dict(x=1.02, y=1, xanchor="left", font=dict(size=11)),
    margin=dict(l=70, r=150, t=95, b=70),
)
fig_cross_feat.show()

## 7. True-Graph Alignment

### 7a — Sign Alignment vs True DAG  
Mean fraction of instances where the causal method's SHAP sign matches the True-graph output per feature.  
Higher = more aligned with the oracle graph.

### 7b — Magnitude TGA vs True DAG  
Mean relative magnitude deviation vs True-graph output per feature.  
Lower = closer to oracle magnitude.

In [13]:
# ── 7a: Sign Alignment heatmap — method × graph ──────────────────────────────
pivot_sa = df_sa.pivot_table(index="Method", columns="Graph", values="MeanSignAlign")

fig_sa = go.Figure(go.Heatmap(
    z            = pivot_sa.values,
    x            = pivot_sa.columns.tolist(),
    y            = pivot_sa.index.tolist(),
    colorscale   = "Blues",
    zmin=0.5, zmax=1.0,
    text         = [[f"{v:.3f}" if not np.isnan(v) else "—" for v in row] for row in pivot_sa.values],
    texttemplate = "%{text}",
    textfont     = dict(size=12),
    colorbar     = dict(title=dict(text="Sign Align", font=dict(size=11))),
    hovertemplate="<b>%{y}</b> — %{x}<br>Sign Align=%{z:.3f}<extra></extra>",
))
fig_sa.update_layout(
    title=dict(text="Sign Alignment vs True DAG — Sachs Real Data<br>"
               "<sup>Fraction of instances where sign matches oracle</sup>",
               font=dict(size=12)),
    height=300, width=600,
    margin=dict(l=130, r=80, t=70, b=60),
    xaxis=dict(tickfont=dict(size=11)),
    yaxis=dict(tickfont=dict(size=11)),
)
fig_sa.show()

In [14]:
# ── 7b: Magnitude TGA heatmap — method × graph ────────────────────────────────
pivot_tga = df_tga.pivot_table(index="Method", columns="Graph", values="MeanTGA")

abs_max_tga = float(np.nanmax(np.abs(pivot_tga.values)))

fig_tga = go.Figure(go.Heatmap(
    z            = pivot_tga.values,
    x            = pivot_tga.columns.tolist(),
    y            = pivot_tga.index.tolist(),
    colorscale   = "RdBu",
    zmid=0,
    zmin=-abs_max_tga, zmax=abs_max_tga,
    text         = [[f"{v:.4f}" if not np.isnan(v) else "—" for v in row] for row in pivot_tga.values],
    texttemplate = "%{text}",
    textfont     = dict(size=12),
    colorbar     = dict(title=dict(text="Mean TGA", font=dict(size=11))),
    hovertemplate="<b>%{y}</b> — %{x}<br>TGA=%{z:.4f}<extra></extra>",
))
fig_tga.update_layout(
    title=dict(text="Magnitude TGA vs True DAG — Sachs Real Data<br>"
               "<sup>Signed: positive = overestimates True; negative = underestimates; white ≈ 0 = matches</sup>",
               font=dict(size=12)),
    height=300, width=600,
    margin=dict(l=130, r=80, t=70, b=60),
    xaxis=dict(tickfont=dict(size=11)),
    yaxis=dict(tickfont=dict(size=11)),
)
fig_tga.show()

In [15]:
# ── 7c: Summary scatter — TGA vs Sign Disagreement (True DAG) ────────────────
# Color = Method, Shape = Graph
# X-axis: TGA, Y-axis: 1 - Sign Alignment (sign disagreement rate)
GRAPH_MARKERS = {"PC": "circle", "LiNGAM": "diamond"}

fig_summary_true = go.Figure()
SEEN_METHODS_TRUE = set()

for _, row in df_sa.merge(df_tga, on=["Method", "Graph"]).iterrows():
    meth = row["Method"]
    g    = row["Graph"]
    show_meth = meth not in SEEN_METHODS_TRUE
    SEEN_METHODS_TRUE.add(meth)
    
    sign_disagree = 1 - row["MeanSignAlign"]
    
    fig_summary_true.add_trace(go.Scatter(
        x    = [row["MeanTGA"]],
        y    = [sign_disagree],
        mode = "markers",
        name = meth,
        legendgroup = meth,
        showlegend  = show_meth,
        marker = dict(
            color  = METHOD_COLORS[meth],
            symbol = GRAPH_MARKERS[g],
            size   = 13,
            line   = dict(width=1.2, color="white"),
        ),
        hovertemplate=(
            f"<b>{meth}</b> — {g}<br>"
            "TGA: %{x:.4f}<br>"
            "Sign disagreement (1−align): %{y:.3f}<extra></extra>"
        ),
    ))

# Graph shape legend (invisible traces)
for g_label, symbol in GRAPH_MARKERS.items():
    fig_summary_true.add_trace(go.Scatter(
        x=[None], y=[None], mode="markers",
        name=g_label,
        legendgroup=f"graph_{g_label}",
        showlegend=True,
        marker=dict(color="gray", symbol=symbol, size=11),
    ))

# Reference line at TGA = 0
fig_summary_true.add_vline(x=0, line=dict(dash="dot", color="lightgray", width=1.5))
fig_summary_true.add_annotation(
    x=0, y=0.48, text="TGA = 0 (matches True)", showarrow=False,
    font=dict(size=9, color="#aaaaaa"), xanchor="center", yanchor="bottom",
)

fig_summary_true.update_layout(
    title=dict(
        text="True-Graph Alignment Summary — Sachs Real Data<br>"
             "<sup>Colour = method, Shape = graph | Lower disagreement + TGA near 0 = closer to True oracle</sup>",
        font=dict(size=12),
    ),
    xaxis=dict(
        title="Magnitude TGA vs True DAG",
        zeroline=True, zerolinewidth=1.5, zerolinecolor="gray",
    ),
    yaxis=dict(
        title="1 − Sign Alignment (sign disagreement rate)",
        range=[0, 0.5],
    ),
    height=480, width=660,
    legend=dict(x=1.02, y=1, xanchor="left", font=dict(size=11)),
    margin=dict(l=70, r=180, t=90, b=70),

)
fig_summary_true.show()

In [16]:
# ── Summary Table: Mean Metrics by Method & Graph (True DAG) ─────────────────
df_summary_true_agg = df_sa.merge(df_tga, on=["Method", "Graph"])
df_summary_true_agg["SignDisagreement"] = 1 - df_summary_true_agg["MeanSignAlign"]
df_summary_true_agg = df_summary_true_agg[["Method", "Graph", "MeanSignAlign", "MeanTGA", "SignDisagreement"]]
df_summary_true_agg = df_summary_true_agg.rename(columns={
    "MeanSignAlign": "Mean Sign Alignment",
    "MeanTGA": "Mean TGA",
    "SignDisagreement": "Mean Sign Disagreement (1−SA)"
})
df_summary_true_agg = df_summary_true_agg.sort_values(["Graph", "Method"])

print("\nSummary: Mean Metrics vs True DAG — Sachs Real Data")
print("=" * 80)
display(df_summary_true_agg.round(4))


Summary: Mean Metrics vs True DAG — Sachs Real Data


,Method,Graph,Mean Sign Alignment,Mean TGA,Mean Sign Disagreement (1−SA)
1,Asymmetric,LiNGAM,0.7660,0.0398,0.2340
3,Causal,LiNGAM,0.6770,0.0359,0.3230
5,Flow,LiNGAM,0.7619,0.2736,0.2381
0,Asymmetric,PC,0.7580,0.2173,0.2420
2,Causal,PC,0.6370,-0.6823,0.3630
4,Flow,PC,0.7164,0.6877,0.2836


## 8. Methods vs Traditional Baseline

Compares each (method × graph) against **Traditional** (graph-free SHAP) using sign-alignment and magnitude TGA metrics.

**Note:** Metrics were already computed in section 2. The DataFrames `df_sa_scratch` and `df_tga_scratch` contain:
- **Sign Alignment vs Traditional** — fraction of (instance, feature) pairs where sign matches Traditional
- **Magnitude TGA vs Traditional** — relative magnitude deviation from Traditional

In [17]:
# ── Metrics already computed in section 2 ─────────────────────────────────────
# Using df_sa_scratch and df_tga_scratch for visualizations

print(f"Traditional baseline metrics (computed in section 2):")
print(f"  Sign Alignment vs Scratch: {len(df_sa_scratch)} rows")
print(f"  TGA vs Scratch: {len(df_tga_scratch)} rows")
print("\nSign Alignment vs Traditional:")
print(df_sa_scratch.pivot_table(index="Method", columns="Graph", values="MeanSignAlign").round(3))

Traditional baseline metrics (computed in section 2):
  Sign Alignment vs Scratch: 6 rows
  TGA vs Scratch: 6 rows

Sign Alignment vs Traditional:
Graph       LiNGAM     PC
Method                   
Asymmetric   0.830  0.832
Causal       0.669  0.687
Flow         0.585  0.598


In [18]:
# ── 8a: Sign Alignment vs Traditional heatmap ─────────────────────────────────
pivot_sa_scratch = df_sa_scratch.pivot_table(
    index="Method", columns="Graph", values="MeanSignAlign"
)

fig_sa_scratch = go.Figure(go.Heatmap(
    z            = pivot_sa_scratch.values,
    x            = pivot_sa_scratch.columns.tolist(),
    y            = pivot_sa_scratch.index.tolist(),
    colorscale   = "Blues",
    zmin=0.5, zmax=1.0,
    text         = [[f"{v:.3f}" if not np.isnan(v) else "—" for v in row]
                    for row in pivot_sa_scratch.values],
    texttemplate = "%{text}",
    textfont     = dict(size=12),
    colorbar     = dict(title=dict(text="Sign Align", font=dict(size=10))),
    hovertemplate="<b>%{y}</b> — %{x}<br>Sign Align=%{z:.3f}<extra></extra>",
))
fig_sa_scratch.update_layout(
    title=dict(
        text="Sign Alignment vs Traditional — Sachs Real Data<br>"
             "<sup>Fraction of instances/features where sign agrees with graph-free output</sup>",
        font=dict(size=13),
    ),
    height=300, width=600,
    margin=dict(l=130, r=80, t=80, b=60),
    xaxis=dict(tickfont=dict(size=11)),
    yaxis=dict(tickfont=dict(size=11)),
)
fig_sa_scratch.show()

In [19]:
# ── 8b: Magnitude TGA vs Traditional heatmap ──────────────────────────────────
pivot_tga_scratch = df_tga_scratch.pivot_table(
    index="Method", columns="Graph", values="MeanTGA"
)

abs_max_scratch = float(np.nanmax(np.abs(pivot_tga_scratch.values)))

fig_tga_scratch = go.Figure(go.Heatmap(
    z            = pivot_tga_scratch.values,
    x            = pivot_tga_scratch.columns.tolist(),
    y            = pivot_tga_scratch.index.tolist(),
    colorscale   = "RdBu",
    zmid=0,
    zmin=-abs_max_scratch, zmax=abs_max_scratch,
    text         = [[f"{v:.4f}" if not np.isnan(v) else "—" for v in row]
                    for row in pivot_tga_scratch.values],
    texttemplate = "%{text}",
    textfont     = dict(size=12),
    colorbar     = dict(title=dict(text="TGA", font=dict(size=10))),
    hovertemplate="<b>%{y}</b> — %{x}<br>TGA=%{z:.4f}<extra></extra>",
))
fig_tga_scratch.update_layout(
    title=dict(
        text="Magnitude TGA vs Traditional — Sachs Real Data<br>"
             "<sup>Signed: positive = overestimates Traditional; negative = underestimates</sup>",
        font=dict(size=13),
    ),
    height=300, width=600,
    margin=dict(l=130, r=80, t=80, b=60),
    xaxis=dict(tickfont=dict(size=11)),
    yaxis=dict(tickfont=dict(size=11)),
)
fig_tga_scratch.show()

In [20]:
# ── 8c: Summary scatter — TGA vs Sign Disagreement (Traditional baseline) ───
# X-axis: TGA, Y-axis: 1 - Sign Alignment (sign disagreement rate)
fig_summary_scratch = go.Figure()
SEEN_METHODS_SCRATCH = set()

for _, row in df_sa_scratch.merge(df_tga_scratch, on=["Method", "Graph"]).iterrows():
    meth = row["Method"]
    g    = row["Graph"]
    show_meth = meth not in SEEN_METHODS_SCRATCH
    SEEN_METHODS_SCRATCH.add(meth)
    
    sign_disagree = 1 - row["MeanSignAlign"]
    
    fig_summary_scratch.add_trace(go.Scatter(
        x    = [row["MeanTGA"]],
        y    = [sign_disagree],
        mode = "markers",
        name = meth,
        legendgroup = meth,
        showlegend  = show_meth,
        marker = dict(
            color  = METHOD_COLORS[meth],
            symbol = GRAPH_MARKERS[g],
            size   = 13,
            line   = dict(width=1.2, color="white"),
        ),
        hovertemplate=(
            f"<b>{meth}</b> — {g}<br>"
            "TGA: %{x:.4f}<br>"
            "Sign disagreement (1−align): %{y:.3f}<extra></extra>"
        ),
    ))

# Graph shape legend
for g_label, symbol in GRAPH_MARKERS.items():
    fig_summary_scratch.add_trace(go.Scatter(
        x=[None], y=[None], mode="markers",
        name=g_label,
        legendgroup=f"graph_{g_label}",
        showlegend=True,
        marker=dict(color="gray", symbol=symbol, size=11),
    ))

# Reference line at TGA = 0
fig_summary_scratch.add_vline(x=0, line=dict(dash="dot", color="lightgray", width=1.5))
fig_summary_scratch.add_annotation(
    x=0, y=0.48, text="TGA = 0 (matches Traditional)", showarrow=False,
    font=dict(size=9, color="#aaaaaa"), xanchor="center", yanchor="bottom",
)

fig_summary_scratch.update_layout(
    title=dict(
        text="Traditional Baseline Alignment Summary — Sachs Real Data<br>"
             "<sup>Colour = method, Shape = graph | Lower disagreement + TGA near 0 = closer to Traditional baseline</sup>",
        font=dict(size=12),
    ),
    xaxis=dict(
        title="Magnitude TGA vs Traditional",
        zeroline=True, zerolinewidth=1.5, zerolinecolor="gray",
    ),
    yaxis=dict(
        title="1 − Sign Alignment (sign disagreement rate)",
        range=[0, 0.5],
    ),
    height=480, width=660,
    legend=dict(x=1.02, y=1, xanchor="left", font=dict(size=11)),
    margin=dict(l=70, r=180, t=90, b=70),

)
fig_summary_scratch.show()

In [21]:
# ── Summary Table: Mean Metrics by Method & Graph (Traditional baseline) ─────
df_summary_scratch_agg = df_sa_scratch.merge(df_tga_scratch, on=["Method", "Graph"])
df_summary_scratch_agg["SignDisagreement"] = 1 - df_summary_scratch_agg["MeanSignAlign"]
df_summary_scratch_agg = df_summary_scratch_agg[["Method", "Graph", "MeanSignAlign", "MeanTGA", "SignDisagreement"]]
df_summary_scratch_agg = df_summary_scratch_agg.rename(columns={
    "MeanSignAlign": "Mean Sign Alignment",
    "MeanTGA": "Mean TGA",
    "SignDisagreement": "Mean Sign Disagreement (1−SA)"
})
df_summary_scratch_agg = df_summary_scratch_agg.sort_values(["Graph", "Method"])

print("\nSummary: Mean Metrics vs Traditional Baseline — Sachs Real Data")
print("=" * 80)
display(df_summary_scratch_agg.round(4))


Summary: Mean Metrics vs Traditional Baseline — Sachs Real Data


,Method,Graph,Mean Sign Alignment,Mean TGA,Mean Sign Disagreement (1−SA)
1,Asymmetric,LiNGAM,0.830,0.2350,0.170
3,Causal,LiNGAM,0.669,2.1182,0.331
5,Flow,LiNGAM,0.585,0.1572,0.415
0,Asymmetric,PC,0.832,0.4125,0.168
2,Causal,PC,0.687,1.4000,0.313
4,Flow,PC,0.598,0.5713,0.402


## 9. Comprehensive Summary Table

All metrics in one table.

In [22]:
rows_summary = []

for meth in BASE_METHODS:
    for g in DISC_GRAPHS:
        key = f"{meth} ({g})"
        if key not in shap_data:
            continue
        ma = mean_abs_shap(shap_data, key)
        rho, _ = spearmanr(ma, scratch_ma)

        j_k5 = top_k_jaccard(ma, scratch_ma, 5) if 5 in K_VALUES else np.nan
        prec_k5_set = set(np.argsort(ma)[-5:]) if 5 in K_VALUES else set()
        prec_k5 = len(prec_k5_set & y_parent_set) / 5 if 5 in K_VALUES else np.nan

        # GSS
        gss_row = df_gss[df_gss["Method"] == meth]
        gss_val = float(gss_row["MeanGSS"].values[0]) if not gss_row.empty else np.nan

        # SSS
        sss_row = df_sss[df_sss["Method"] == meth]
        sss_val = float(sss_row["MeanSSS"].values[0]) if not sss_row.empty else np.nan

        # Sign Alignment
        sa_row = df_sa[(df_sa["Method"] == meth) & (df_sa["Graph"] == g)]
        sa_val = float(sa_row["MeanSignAlign"].values[0]) if not sa_row.empty else np.nan

        # TGA
        tga_row = df_tga[(df_tga["Method"] == meth) & (df_tga["Graph"] == g)]
        tga_val = float(tga_row["MeanTGA"].values[0]) if not tga_row.empty else np.nan

        rows_summary.append(dict(
            Method     = meth,
            Graph      = g,
            Spearman_ρ = round(float(rho), 3),
            Jaccard_5  = round(float(j_k5), 3) if not np.isnan(j_k5) else np.nan,
            Prec_5     = round(float(prec_k5), 3) if not np.isnan(prec_k5) else np.nan,
            Rand_Base  = round(float(random_base), 3),
            GSS        = round(float(gss_val), 4) if not np.isnan(gss_val) else np.nan,
            SSS        = round(float(sss_val), 4) if not np.isnan(sss_val) else np.nan,
            SignAlign  = round(float(sa_val), 3)  if not np.isnan(sa_val) else np.nan,
            MagTGA     = round(float(tga_val), 4) if not np.isnan(tga_val) else np.nan,
        ))

df_summary = (
    pd.DataFrame(rows_summary)
    .sort_values(["Graph", "Method"])
    .reset_index(drop=True)
)

pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

print(f"\nTotal rows: {len(df_summary)}")
df_summary


Total rows: 6


,Method,Graph,Spearman_ρ,Jaccard_5,Prec_5,Rand_Base,GSS,SSS,SignAlign,MagTGA
0,Asymmetric,LiNGAM,0.9880,0.6670,1.0000,1.0000,0.1775,0.8140,0.7660,0.0398
1,Causal,LiNGAM,0.2850,0.2500,1.0000,1.0000,-0.7182,0.6820,0.6770,0.0359
2,Flow,LiNGAM,0.3090,0.4290,1.0000,1.0000,0.4141,0.7812,0.7620,0.2736
3,Asymmetric,PC,0.7820,0.6670,1.0000,1.0000,0.1775,0.8140,0.7580,0.2173
4,Causal,PC,0.5150,0.4290,1.0000,1.0000,-0.7182,0.6820,0.6370,-0.6823
5,Flow,PC,0.4910,0.4290,1.0000,1.0000,0.4141,0.7812,0.7160,0.6877


## 10. Per-Feature Diagnostics

Surface features at both extremes of each metric for targeted qualitative analysis.

**K_DIAG = 3** (smaller than synthetic data due to only 10 features total).

In [23]:
K_DIAG = 3

In [24]:
# ── 10b: Top-K features by GSS ────────────────────────────────────────────────
gss_feat = compute_gss(shap_data, BASE_METHODS)

print(f"\n{'═' * 72}")
print(f"  Top-{K_DIAG} features by |GSS|  (highest magnitude instability PC vs LiNGAM)")
print(f"  Positive GSS = PC assigns higher importance;  Negative GSS = LiNGAM higher")
print(f"{'═' * 72}")
for meth, arr in gss_feat.items():
    df_g = top_k_features(arr, feature_names, k=K_DIAG, ascending=False, score_col="GSS")
    print(f"\n  {meth}")
    display(df_g)


════════════════════════════════════════════════════════════════════════
  Top-3 features by |GSS|  (highest magnitude instability PC vs LiNGAM)
  Positive GSS = PC assigns higher importance;  Negative GSS = LiNGAM higher
════════════════════════════════════════════════════════════════════════

  Asymmetric


,feature,GSS
rank,,
1,erk,2.7701
2,mek,-2.3836
3,pip3,0.8083



  Causal


,feature,GSS
rank,,
1,p38,-14.4054
2,erk,12.4435
3,pka,-11.5649



  Flow


,feature,GSS
rank,,
1,erk,36.3836
2,pka,-23.8241
3,pkc,-7.9722


In [25]:
# ── 10c: Bottom-K features by SSS ─────────────────────────────────────────────
# Using sss_feat computed in section 2

print(f"\n{'═' * 72}")
print(f"  Bottom-{K_DIAG} features by SSS  (lower = more sign-unstable)")
print(f"{'═' * 72}")
for meth, arr in sss_feat.items():
    df_s = top_k_features(arr, feature_names, k=K_DIAG, ascending=True, score_col="SSS")
    print(f"\n  {meth}")
    display(df_s)


════════════════════════════════════════════════════════════════════════
  Bottom-3 features by SSS  (lower = more sign-unstable)
════════════════════════════════════════════════════════════════════════

  Asymmetric


,feature,SSS
rank,,
1,pip2,0.5700
2,pkc,0.5800
3,pip3,0.6700



  Causal


,feature,SSS
rank,,
1,pkc,0.3400
2,p38,0.4900
3,pip3,0.5200



  Flow


,feature,SSS
rank,,
1,pkc,0.4948
2,p38,0.6200
3,mek,0.7526


In [26]:
# ── 10d: Bottom-K features by Sign Alignment vs True DAG ──────────────────────
# Using sign_align computed in section 2

print(f"\n{'═' * 72}")
print(f"  Bottom-{K_DIAG} features by Sign Alignment vs True DAG")
print(f"  (lower = method/graph attributions most often flip sign vs True-DAG oracle)")
print(f"{'═' * 72}")
for (meth, g), arr in sign_align.items():
    df_sa_f = top_k_features(arr, feature_names, k=K_DIAG, ascending=True, score_col="SignAlign")
    print(f"\n  {meth} ({g})")
    display(df_sa_f)


════════════════════════════════════════════════════════════════════════
  Bottom-3 features by Sign Alignment vs True DAG
  (lower = method/graph attributions most often flip sign vs True-DAG oracle)
════════════════════════════════════════════════════════════════════════

  Asymmetric (PC)


,feature,SignAlign
rank,,
1,plc,0.4900
2,p38,0.5800
3,pkc,0.6700



  Asymmetric (LiNGAM)


,feature,SignAlign
rank,,
1,plc,0.4800
2,pip2,0.4800
3,pkc,0.6300



  Causal (PC)


...

,feature,SignAlign
rank,,
1,pip3,0.4700
2,plc,0.5300
3,pip2,0.5600



  Flow (PC)


,feature,SignAlign
rank,,
1,pkc,0.5900
2,plc,0.6263
3,pka,0.6500



  Flow (LiNGAM)


,feature,SignAlign
rank,,
1,pkc,0.5400
2,pip2,0.5684
3,p38,0.6111


In [27]:
# ── 10e: Top-K features by Magnitude TGA vs True DAG ──────────────────────────
# Using tga_data computed in section 2

print(f"\n{'═' * 72}")
print(f"  Top-{K_DIAG} features by Magnitude TGA vs True DAG")
print(f"  (higher = absolute magnitudes furthest from the True-DAG oracle)")
print(f"{'═' * 72}")
for (meth, g), arr in tga_data.items():
    df_t = top_k_features(np.abs(arr), feature_names, k=K_DIAG, ascending=False, score_col="|TGA|")
    print(f"\n  {meth} ({g})")
    display(df_t)


════════════════════════════════════════════════════════════════════════
  Top-3 features by Magnitude TGA vs True DAG
  (higher = absolute magnitudes furthest from the True-DAG oracle)
════════════════════════════════════════════════════════════════════════

  Asymmetric (PC)


,feature,|TGA|
rank,,
1,erk,4.2081
2,mek,3.4345
3,pka,0.9119



  Asymmetric (LiNGAM)


,feature,|TGA|
rank,,
1,erk,1.4380
2,mek,1.0509
3,plc,0.6610



  Causal (PC)


...

,feature,|TGA|
rank,,
1,p38,14.9764
2,erk,5.7229
3,pka,4.8970



  Flow (PC)


,feature,|TGA|
rank,,
1,erk,31.4195
2,pka,18.6991
3,pkc,11.4384



  Flow (LiNGAM)


,feature,|TGA|
rank,,
1,p38,11.8055
2,pka,5.1249
3,erk,4.9641


In [28]:
# ── 10e: Top-K features by Magnitude TGA vs Traditional Shapley ──────────────────────────
# Using tga_scratch computed in section 2

print(f"\n{'═' * 72}")
print(f"  Top-{K_DIAG} features by Magnitude TGA vs Traditional Shapley")
print(f"  (higher = absolute magnitudes furthest from the Traditional Shapley oracle)")
print(f"{'═' * 72}")
for (meth, g), arr in tga_scratch.items():
    df_t = top_k_features(np.abs(arr), feature_names, k=K_DIAG, ascending=False, score_col="|TGA|")
    print(f"\n  {meth} ({g})")
    display(df_t)


════════════════════════════════════════════════════════════════════════
  Top-3 features by Magnitude TGA vs Traditional Shapley
  (higher = absolute magnitudes furthest from the Traditional Shapley oracle)
════════════════════════════════════════════════════════════════════════

  Asymmetric (PC)


,feature,|TGA|
rank,,
1,pka,2.3533
2,erk,1.9219
3,mek,1.0635



  Asymmetric (LiNGAM)


,feature,|TGA|
rank,,
1,pka,2.0678
2,mek,1.3201
3,erk,0.8482



  Causal (PC)


...

,feature,|TGA|
rank,,
1,p38,16.2389
2,pka,9.3743
3,erk,7.4272



  Flow (PC)


,feature,|TGA|
rank,,
1,pka,12.0853
2,raf,9.6458
3,erk,5.7823



  Flow (LiNGAM)


,feature,|TGA|
rank,,
1,erk,30.6013
2,pka,11.7388
3,p38,11.2258


## 11. DAG Node Highlight

Highlight a specific feature node in PC, LiNGAM, and True graphs to visualize its causal role.

**Blue** arrows = incoming edges (parents)  
**Orange-red** arrows = outgoing edges (children)

In [29]:
# ── Load DAG graphs for sachs dataset ─────────────────────────────────────────
import matplotlib.pyplot as plt

pc_path     = CAUSAL_DIR / f"{DATASET}_pc_results.json"
lingam_path = CAUSAL_DIR / f"{DATASET}_lingam_results.json"
true_path   = CAUSAL_DIR / f"{DATASET}_true_full_adjacency.npy"

with open(pc_path) as f:
    pc_res = json.load(f)
with open(lingam_path) as f:
    lg_res = json.load(f)

pc_adj     = np.array(pc_res["adjacency_matrix"])
lingam_adj = np.array(lg_res["adjacency_matrix"])
true_adj   = np.load(true_path)
names      = pc_res["feature_names"]

G_pc,     td_pc,     lvg_pc,     src_pc,     y_pc,     ml_pc     = build_dag_graph(pc_adj,     names)
G_lingam, td_lingam, lvg_lingam, src_lingam, y_lingam, ml_lingam = build_dag_graph(lingam_adj, names)
G_true,   td_true,   lvg_true,   src_true,   y_true,   ml_true   = build_dag_graph(true_adj,   names)

pos_pc     = make_dag_pos(lvg_pc,     ml_pc)
pos_lingam = make_dag_pos(lvg_lingam, ml_lingam)
pos_true   = make_dag_pos(lvg_true,   ml_true)

print(f"Graph stats:")
print(f"  PC: {G_pc.number_of_edges()} edges")
print(f"  LiNGAM: {G_lingam.number_of_edges()} edges")
print(f"  True: {G_true.number_of_edges()} edges")

Graph stats:
  PC: 29 edges
  LiNGAM: 36 edges
  True: 27 edges


In [30]:
def plot_dag_highlight_real(feature_name: str, figsize=(26, 10)):
    """
    Draw PC, LiNGAM, and True DAGs side by side with one feature node highlighted.

    Parameters
    ----------
    feature_name : str
        Name of the feature to highlight, e.g. "pip3", "erk".
    figsize : tuple
        Figure size (width, height).
    """
    fig, axes = plt.subplots(1, 3, figsize=figsize)
    fig.patch.set_facecolor("#f0f0f0")

    draw_dag_highlight(
        axes[0], G_pc, pos_pc, td_pc, src_pc, y_pc, ml_pc,
        title=f"PC Graph — Sachs  ·  highlight: {feature_name}",
        highlight_name=feature_name,
        names=names,
    )
    draw_dag_highlight(
        axes[1], G_lingam, pos_lingam, td_lingam, src_lingam, y_lingam, ml_lingam,
        title=f"LiNGAM Graph — Sachs  ·  highlight: {feature_name}",
        highlight_name=feature_name,
        names=names,
    )
    draw_dag_highlight(
        axes[2], G_true, pos_true, td_true, src_true, y_true, ml_true,
        title=f"True Graph — Sachs  ·  highlight: {feature_name}",
        highlight_name=feature_name,
        names=names,
    )

    fig.suptitle(
        f"DAG Node Highlight — {feature_name} — Sachs Real Data\n"
        f"Blue = incoming edges (parents of {feature_name})  ·  "
        f"Orange-red = outgoing edges (children of {feature_name})",
        fontsize=13, fontweight="bold", y=0.99,
    )
    plt.tight_layout(rect=[0, 0.02, 1, 0.97])
    plt.show()


# ── Example usage ─────────────────────────────────────────────────────────────
# plot_dag_highlight_real("pip3")
# plot_dag_highlight_real("erk")

## 12. Export Summary Table

In [31]:
# Export the summary table to CSV
out_path = BASE_DIR / "data" / "explainability" / "real_data_summary_metrics.csv"
df_summary.to_csv(out_path, index=False)
print(f"Saved → {out_path}")

Saved → /Users/juanrios/Documents/master_thesis/data/explainability/real_data_summary_metrics.csv
